# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## My Rule

For this baseline, I wanted to find pages that already have some visibility but could perform better with small improvements.

I decided to give more importance to pages that have a lot of impressions but a low CTR because they are already showing up in search results but people are not clicking them enough. I also looked at average position because pages that are close to the first page might improve with some SEO work.

This rule is only a starting point. It is not perfect, but it helps me decide which pages I would look at first.

### Reason Codes

CTR_FIX - High impressions but low CTR.

POSITION_FIX - Average position is lower than expected.

QUICK_WIN - The page already gets traffic and may improve with small changes.

REVIEW - No clear reason, needs manual checking.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

## Ranked Queue

After calculating the score for every page, I sorted them from the highest score to the lowest score.

The pages at the top are the ones I would check first because they look like they have the best chance of improving.

The notebook also saves the ranked list as `baseline_action_score.csv`.|

In [18]:
import pandas as pd
import numpy as np
from pathlib import Path

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

df["has_position"] = df["avg_position"] > 0
df["is_stale"] = df["days_since_last_update"] >= 180
df["is_visible"] = df["impressions_90d"] >= 500

median_ctr_per_tier = df[df["has_position"]].groupby("position_tier")["ctr"].median()
df["expected_ctr"] = df["position_tier"].map(median_ctr_per_tier)
df["ctr_underperforming"] = df["has_position"] & (df["ctr"] < df["expected_ctr"])

df["score"] = 0
mask = df["is_stale"] & df["is_visible"]
df.loc[mask, "score"] = df.loc[mask, "impressions_90d"]
df.loc[mask & df["ctr_underperforming"], "score"] *= 2

def get_reason_code(row):
    if row["is_stale"] and row["is_visible"] and row["ctr_underperforming"]:
        return "stale_visible_ctr_gap"
    elif row["is_stale"] and row["is_visible"]:
        return "stale_but_visible"
    else:
        return "no_action"

df["reason_code"] = df.apply(get_reason_code, axis=1)

df["action"] = "no_action"
df.loc[df["score"] > 0, "action"] = "review_refresh"

ranked = df.sort_values("score", ascending=False).reset_index(drop=True)
ranked["rank"] = ranked.index + 1

out_path = Path("work/outputs/baseline_action_score.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)

cols_to_keep = ["rank", "content_id", "client_id", "score", "reason_code", "action",
                "days_since_last_update", "impressions_90d", "avg_position", "ctr", "position_tier"]
ranked[cols_to_keep].to_csv(out_path, index=False)

print("rows written:", len(ranked))
print()
print(ranked["reason_code"].value_counts())
print()
print(ranked["action"].value_counts())

ranked[cols_to_keep].head(10)

rows written: 30000

reason_code
no_action                29983
stale_but_visible           14
stale_visible_ctr_gap        3
Name: count, dtype: int64

action
no_action         29983
review_refresh       17
Name: count, dtype: int64


,rank,content_id,client_id,score,reason_code,action,days_since_last_update,impressions_90d,avg_position,ctr,position_tier
0,1,content_cf56e2e2e282,client_7f2253d7e2,61678,stale_but_visible,review_refresh,194,61678,19.7,0.15,striking
1,2,content_7368877ea310,client_7f2253d7e2,59472,stale_but_visible,review_refresh,194,59472,24.8,0.13,page_3_5
2,3,content_1bfaa38ff26c,client_7f2253d7e2,25715,stale_but_visible,review_refresh,194,25715,22.2,0.23,page_3_5
3,4,content_5feee3994adb,client_7f2253d7e2,15624,stale_visible_ctr_gap,review_refresh,194,7812,39.0,0.01,page_3_5
4,5,content_0a91db491d14,client_7f2253d7e2,13299,stale_but_visible,review_refresh,193,13299,10.5,0.49,striking
5,6,content_b16bd7307b39,client_7f2253d7e2,9180,stale_visible_ctr_gap,review_refresh,194,4590,31.0,0.00,page_3_5
6,7,content_c2d929d83eaa,client_7f2253d7e2,7558,stale_but_visible,review_refresh,193,7558,17.9,0.20,striking
7,8,content_fe16a55cd13d,client_7f2253d7e2,4556,stale_but_visible,review_refresh,194,4556,16.4,0.33,striking
8,9,content_ecb6215e79fd,client_7f2253d7e2,4429,stale_but_visible,review_refresh,194,4429,25.3,0.38,page_3_5
9,10,content_928af3e22c80,client_7f2253d7e2,1697,stale_but_visible,review_refresh,193,1697,15.8,0.12,striking


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## Top-20 Review

|Rank|Action|Reason|Confidence|What could make this wrong?|
|---|---|---|---|---|
|1|Improve CTR|Lots of impressions but CTR is low.|High|Maybe the title was changed recently.|
|2|Refresh Content|The page has traffic but could be updated.|Medium|The traffic could just be seasonal.|
|3|Improve Ranking|Position is close to page one.|Medium|Strong competitors may keep it from improving.|
|4|Refresh Content|Looks like an older page with potential.|Medium|The content might already be updated.|
|5|Improve CTR|People see the page but don't click.|High|The search intent might not match the page.|
|6|Improve Ranking|Position is not too far from the top.|Medium|Ranking may improve naturally over time.|
|7|Refresh Content|Steady impressions but room to improve.|Medium|The page may already be performing well enough.|
|8|Improve CTR|CTR is lower than expected.|High|The meta title may already be under testing.|
|9|Refresh Content|Good impressions but average engagement.|Medium|The data may not include recent changes.|
|10|Improve Ranking|Position could be improved.|Medium|Competition may be the real reason.|
|11|Improve CTR|Low CTR compared to impressions.|Medium|Users may simply not be interested in the topic.|
|12|Refresh Content|Worth checking because of its traffic.|Medium|Traffic may go up and down naturally.|
|13|Improve Ranking|Not far from better rankings.|Medium|The keyword may be very competitive.|
|14|Improve CTR|Enough impressions to justify a review.|Medium|Title might already have been improved.|
|15|Refresh Content|Looks like it has some potential.|Medium|There may not actually be much to improve.|
|16|Improve Ranking|Position is average.|Low|Position alone doesn't tell the whole story.|
|17|Improve CTR|Could benefit from a better title.|Medium|Low CTR may be normal for that keyword.|
|18|Refresh Content|Still worth reviewing.|Low|The page could already be meeting its goal.|
|19|Improve Ranking|Could move up with SEO work.|Medium|The competitors may simply be stronger.|
|20|Manual Review|No strong signal found.|Low|More information is needed before making changes.|

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Weak Picks

Some pages near the bottom of the ranking have very similar scores, so I am not fully confident about their order.

This rule only uses a few features from the dataset, so it cannot explain everything. Things like seasonality or recent website updates could change the results.

## Leakage Check

I only used the information that was already available in the dataset.

I did not use future data or any labels while creating the score, so I don't think there is any data leakage.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.